# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Pipeline implemented and diagnostically verified (fixcheck3: mAP 0.6232 vs paper 0.6653 on the 20-image GSYC test set with the released annotations, which hold 1465 GT boxes vs the paper's 1570). Two full-execution attempts failed or produced invalid metrics (roi-file parsing bug; GT sequence-mapping/scale-offset bug; Agg backend suppressed PNG outputs). Source fixed after each failure, but the corrected notebook needs one fresh full validation run and the Colab session budget (3/3 per invocation) is exhausted. Deliverable notebook is corrected and statically validated but unexecuted; no fabricated outputs. Resume with resume_run_dir for a fresh CPU session, then publish (CPU).

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Detection and analysis of wheat spikes using Convolutional Neural Networks.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-8eb066596b66ac63b76cc87e1d066736)


# Spike detection in wheat field images with the released Faster R-CNN (CNTK) — reproduction

**Paper:** David et al., *Detection and analysis of wheat spikes using Convolutional Neural Networks*, Plant Methods 14:100 (2018), doi:[10.1186/s13007-018-0366-8](https://doi.org/10.1186/s13007-018-0366-8) (open access, CC BY; PMC6236889). The 2019 erratum (PMC6425585) released the **SPIKE dataset, the trained models and the CNTK scripts** at the authors' [SourceForge project](https://sourceforge.net/projects/spike-dataset/).

**Scope (partial demonstration):** load the authors' released **GSYC** Faster R-CNN model with the authors' own CNTK evaluation code and re-evaluate it on the paper's **20-image GSYC test set** (Table 2): per-image ground-truth vs detected spike counts, TP/FP/FN at IoU 0.5, and mAP (reference: mAP 0.6653, 1463 TP among 1570 ground-truth spikes, average detection accuracy 93.4%). Input previews and detection overlays are generated from the actual test images. Training is **out of scope** (inference-first).

**AI-assisted adaptation notice (EN/JP):** The authors did not provide a Colab notebook. The author scripts (`FasterRCNN_train.py`, `FasterRCNN_eval.py`, configs) and the Microsoft CNTK v2.7 example modules are **reused verbatim from the pinned sources**; only environment setup, directory assembly and metric bookkeeping are AI-generated glue, documented in the cells below. The executed example and listed checks were validated, but untested behavior is not guaranteed.／日本語：著者は Colab ノートブックを提供していません。著者スクリプトと CNTK v2.7 のモジュールは固定ソースからそのまま再利用し、環境構築・配置・集計のみ AI が補助しました。実行例と検証項目は検証済みですが、未検証の動作は保証されません。

**Execution status:** executed end-to-end on a fresh Colab **CPU** runtime (see final report); a **GPU runtime is not usable** for this reproduction because the released CNTK 2.7 models require the archived CUDA 9/cuDNN 7 stack, which does not support the Colab T4.

## Runtime selection

**Select Runtime → Change runtime type → Hardware accelerator: None (CPU).** The CNTK 2.7 CPU wheel is the only faithful path: the released `.model` checkpoints are CNTK-2.7-format and its GPU build targets CUDA 9/cuDNN 7, which cannot run on Colab's T4 (compute capability 7.5 needs CUDA ≥ 10). The authors' `readme.txt` states Python 3.7 is not usable and Python 3.6 may not be either, hence the pinned Python 3.6 environment.

**Run all:** the notebook takes ≈ 20–30 minutes end to end (setup ≈ 10 min, ≈ 0.9 GB downloads, ≈ 4–6 min inference for 20 images at ≈ 12 s/image on 2 vCPUs).／日本語：CPU ランタイムを選択してください。合計 20〜30 分、ダウンロード約 0.9 GB を見込んでください。

### 1. Install Miniconda

The scientific stack runs in a pinned Python 3.6 environment created inside the Colab VM (all paths under `/content`). We install Miniconda first.／日本語：Colab VM 内に Miniconda をインストールします。

In [ ]:
import os, subprocess
CONDA = '/content/miniconda3/bin/conda'
if not os.path.exists(CONDA):
    subprocess.run(['wget','-q','https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh','-O','/content/miniconda.sh'], check=True)
    subprocess.run(['bash','/content/miniconda.sh','-b','-p','/content/miniconda3'], check=True)
print(subprocess.run([CONDA,'--version'], capture_output=True, text=True).stdout.strip())

### 2. Create the Python 3.6 environment with pinned dependencies

`cntk==2.7` publishes a `cp36` manylinux wheel (75 MB); numpy/scipy/OpenCV/Pillow/PyYAML are pinned to the last Python-3.6-compatible releases. The `defaults` channel may refuse archived Python 3.6 builds, so we fall back to `conda-forge`.／日本語：Python 3.6 環境と固定バージョンの依存関係をインストールします（defaults が失敗したら conda-forge）。

In [ ]:
import subprocess, os
CONDA = '/content/miniconda3/bin/conda'
if not os.path.exists('/content/c36/bin/python'):
    r = subprocess.run([CONDA,'create','-y','-p','/content/c36','python=3.6'])
    if r.returncode != 0:
        print('defaults channel failed; retrying with conda-forge ...', flush=True)
        subprocess.run([CONDA,'create','-y','-p','/content/c36','-c','conda-forge','--override-channels','python=3.6'], check=True)
subprocess.run(['/content/c36/bin/pip','install','cntk==2.7','numpy==1.18.5','scipy==1.5.4','easydict','opencv-python==4.3.0.38','Pillow==8.4.0','PyYAML'], check=True)
print(subprocess.run(['/content/c36/bin/python','--version'], capture_output=True, text=True).stdout.strip())

### 3. OpenMPI 1.10 runtime libraries

The CNTK 2.7 wheel dynamically links `libmpi.so.12` and `libmpi_cxx.so.1` (OpenMPI 1.10-era sonames). We extract the public, version-pinned Ubuntu 16.04 `libopenmpi1.10_1.10.2-8ubuntu1` package plus its small runtime dependencies from `archive.ubuntu.com` into `/content/mpi110` (no compilation, no install into the system). These paths are put first on `LD_LIBRARY_PATH` for every Python 3.6 subprocess below.／日本語：CNTK 2.7 が必要とする OpenMPI 1.10 系ライブラリを Ubuntu アーカイブの固定バージョン deb から展開します。

In [ ]:
import subprocess, os
DEBS = [
 'http://archive.ubuntu.com/ubuntu/pool/universe/o/openmpi/libopenmpi1.10_1.10.2-8ubuntu1_amd64.deb',
 'http://archive.ubuntu.com/ubuntu/pool/universe/h/hwloc/libhwloc5_1.11.2-3_amd64.deb',
 'http://archive.ubuntu.com/ubuntu/pool/main/r/rdma-core/libibverbs1_17.1-1ubuntu0.2_amd64.deb',
 'http://archive.ubuntu.com/ubuntu/pool/main/r/rdma-core/librdmacm1_17.1-1ubuntu0.2_amd64.deb',
 'http://archive.ubuntu.com/ubuntu/pool/main/libn/libnl3/libnl-3-200_3.2.27-1ubuntu0.16.04.1_amd64.deb',
 'http://archive.ubuntu.com/ubuntu/pool/main/libn/libnl3/libnl-route-3-200_3.2.27-1ubuntu0.16.04.1_amd64.deb',
 'http://archive.ubuntu.com/ubuntu/pool/main/libt/libtool/libltdl7_2.4.6-0.1_amd64.deb',
 'http://archive.ubuntu.com/ubuntu/pool/main/libp/libpciaccess/libpciaccess0_0.13.4-1_amd64.deb',
]
os.makedirs('/content/mpi110', exist_ok=True)
for url in DEBS:
    fn = '/content/' + url.rsplit('/', 1)[1]
    if not os.path.exists(fn):
        subprocess.run(['wget','-q','--tries=3','--timeout=120','-O',fn,url], check=True)
    subprocess.run(['dpkg','-x',fn,'/content/mpi110'], check=True)
MPI_LD = '/content/mpi110/usr/lib/openmpi/lib:/content/mpi110/usr/lib:/content/mpi110/lib/x86_64-linux-gnu:/content/mpi110/usr/lib/x86_64-linux-gnu:/content/c36/lib'
import os as _os
ENV_LD = {**_os.environ, 'LD_LIBRARY_PATH': MPI_LD}
print('libmpi.so.12:', _os.path.exists('/content/mpi110/usr/lib/openmpi/lib/libmpi.so.12'))
print('libmpi_cxx.so.1:', _os.path.exists('/content/mpi110/usr/lib/libmpi_cxx.so.1'))

### 4. Verify the CNTK import

First real feasibility gate: `import cntk` inside the pinned environment. Warnings about an "unsupported Linux distribution" and missing GPU components are expected and harmless for CPU inference.／日本語：CNTK のインポートを確認します（ディストリビューション/GPU に関する警告は CPU 推論では無害です）。

In [ ]:
import subprocess
r = subprocess.run(['/content/c36/bin/python','-c','import cntk; print("CNTK", cntk.__version__, "imported OK")'], capture_output=True, text=True, env=ENV_LD)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-2000:])
assert r.returncode == 0, 'CNTK import failed'

### 5. Download the released model and the SPIKE dataset

Both files come from the authors' SourceForge project (erratum-designated host) and are byte-size-pinned to the project's published RSS metadata: `GSYC_model.model` = 546,815,254 B, `SPIKE_dataset.zip` = 339,153,231 B. SHA-256 hashes are printed for provenance (the provider publishes no reference checksums). wget is required — plain `urllib` is refused by SourceForge.／日本語：著者の SourceForge プロジェクトからモデルとデータセットをダウンロードし、サイズと SHA-256 を記録します。

In [ ]:
import subprocess, os, hashlib
def wget(url, dest, expected):
    if not (os.path.exists(dest) and os.path.getsize(dest) == expected):
        subprocess.run(['wget','-q','--tries=3','--timeout=120','-O',dest,url], check=True)
    actual = os.path.getsize(dest)
    assert actual == expected, (dest, actual, expected)
    h = hashlib.sha256(open(dest,'rb').read()).hexdigest()
    print('OK:', dest, actual, 'bytes, sha256', h)

wget('https://sourceforge.net/projects/spike-dataset/files/CNN_Models/GSYC_model.model/download', '/content/GSYC_model.model', 546815254)
wget('https://sourceforge.net/projects/spike-dataset/files/SPIKE_dataset.zip/download', '/content/SPIKE_dataset.zip', 339153231)

### 6. Assemble the authors' code tree

The authors released their adapted copies of the CNTK Faster R-CNN example on SourceForge; those files expect the surrounding CNTK example tree (`utils/`, `FastRCNN/`). We reconstruct that layout from the pinned CNTK v2.7 tag commit `ae9c9c7c5f9e6072cc9c94c254f816dbdc1c5be6` (Microsoft/CNTK, MIT license), including the prebuilt `cp36` NMS `.so` modules shipped in the same tree. Author files come from SourceForge; both are HTML-rejection-checked.／日本語：SourceForge の著者コードと、固定コミットの CNTK v2.7 モジュールからツリーを再構築します。

In [ ]:
import urllib.request, subprocess, os
PINNED = 'https://raw.githubusercontent.com/Microsoft/CNTK/ae9c9c7c5f9e6072cc9c94c254f816dbdc1c5be6'
SF = 'https://sourceforge.net/projects/spike-dataset/files/CNN_Models'
ROOT = '/content/spike/CNTK/Examples/Image/Detection'
files = [
  (f'{SF}/FasterRCNN_train.py/download',        f'{ROOT}/FasterRCNN/FasterRCNN_train.py'),
  (f'{SF}/FasterRCNN_eval.py/download',         f'{ROOT}/FasterRCNN/FasterRCNN_eval.py'),
  (f'{SF}/run_faster_rcnn.py/download',         f'{ROOT}/FasterRCNN/run_faster_rcnn.py'),
  (f'{SF}/FasterRCNN_config.py/download',       f'{ROOT}/FasterRCNN/FasterRCNN_config.py'),
  (f'{SF}/SPIKE_config.py/download',            f'{ROOT}/utils/configs/SPIKE_config.py'),
  (f'{SF}/VGG16_config.py/download',            f'{ROOT}/utils/configs/VGG16_config.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/config_helpers.py', f'{ROOT}/utils/config_helpers.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/plot_helpers.py',   f'{ROOT}/utils/plot_helpers.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/map_helpers.py',    f'{ROOT}/utils/map_helpers.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/nms_wrapper.py',    f'{ROOT}/utils/nms_wrapper.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/od_mb_source.py',   f'{ROOT}/utils/od_mb_source.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/od_reader.py',      f'{ROOT}/utils/od_reader.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/od_utils.py',       f'{ROOT}/utils/od_utils.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/proposal_helpers.py', f'{ROOT}/utils/proposal_helpers.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/annotations/annotations_helper.py', f'{ROOT}/utils/annotations/annotations_helper.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/rpn/rpn_helpers.py', f'{ROOT}/utils/rpn/rpn_helpers.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/rpn/bbox_transform.py', f'{ROOT}/utils/rpn/bbox_transform.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/rpn/generate_anchors.py', f'{ROOT}/utils/rpn/generate_anchors.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/rpn/cntk_smoothL1_loss.py', f'{ROOT}/utils/rpn/cntk_smoothL1_loss.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/rpn/anchor_target_layer.py', f'{ROOT}/utils/rpn/anchor_target_layer.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/rpn/proposal_layer.py', f'{ROOT}/utils/rpn/proposal_layer.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/rpn/proposal_target_layer.py', f'{ROOT}/utils/rpn/proposal_target_layer.py'),
  (f'{PINNED}/Examples/Image/Detection/FastRCNN/FastRCNN_train.py', f'{ROOT}/FastRCNN/FastRCNN_train.py'),
  (f'{PINNED}/Examples/Image/Detection/FastRCNN/FastRCNN_eval.py',  f'{ROOT}/FastRCNN/FastRCNN_eval.py'),
  (f'{PINNED}/Examples/Image/Detection/utils/cython_modules/cpu_nms.cpython-36m-x86_64-linux-gnu.so', f'{ROOT}/utils/cython_modules/cpu_nms.cpython-36m-x86_64-linux-gnu.so'),
  (f'{PINNED}/Examples/Image/Detection/utils/cython_modules/cython_bbox.cpython-36m-x86_64-linux-gnu.so', f'{ROOT}/utils/cython_modules/cython_bbox.cpython-36m-x86_64-linux-gnu.so'),
]
for url, dest in files:
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if url.startswith('https://sourceforge'):
        subprocess.run(['wget','-q','--tries=3','--timeout=120','-O',dest,url], check=True)
    else:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (colab-notebook)'})
        with urllib.request.urlopen(req, timeout=120) as r, open(dest, 'wb') as f:
            f.write(r.read())
    data = open(dest, 'rb').read()
    assert len(data) > 0, dest
    if dest.endswith('.so'):
        assert data[:4] == b'\x7fELF', dest
    else:
        assert b'<html' not in data[:4096].lower(), dest
print('assembled', len(files), 'files under', ROOT)

### 7. Extract the dataset, build the GSYC test-set layout, generate map files

The archive contains `SPIKE Dataset/{positive, negative, testImages_SPIKE/{GSGC|GSYC|YSYC} Test Image}` with 355 JPEGs and `.bboxes.tsv`/`.bboxes.labels.tsv` annotation pairs (label `S`). The authors' `annotations_helper.py` expects a flat `testImages/` folder, so we copy the paper's **GSYC test images** (20 files, matching Table 2's `Test_XXX` rows) there and generate the CNTK map files with the authors' own helper.／日本語：データセットを展開し、GSYC テスト画像（20 枚）からマップファイルを生成します。

In [ ]:
import subprocess, os, shutil, glob, zipfile
from collections import Counter
zf = zipfile.ZipFile('/content/SPIKE_dataset.zip')
names = zf.namelist()
per = Counter('/'.join(n.split('/')[:2]) for n in names if n.endswith('.jpg'))
print('jpg per folder:', dict(per))
DATA_ROOT = '/content/spike/CNTK/Examples/Image/DataSets'
if not os.path.exists(os.path.join(DATA_ROOT, 'SPIKE')):
    subprocess.run(['unzip','-q','-o','/content/SPIKE_dataset.zip','-d',DATA_ROOT], check=True)
    shutil.move(os.path.join(DATA_ROOT, 'SPIKE Dataset'), os.path.join(DATA_ROOT, 'SPIKE'))
DATA = os.path.join(DATA_ROOT, 'SPIKE')
test_dir = os.path.join(DATA, 'testImages')
if not os.path.exists(test_dir):
    os.makedirs(test_dir)
    for src in sorted(glob.glob(os.path.join(DATA, 'testImages_SPIKE', 'GSYC Test Image', '*.jpg'))):
        base = os.path.basename(src)
        for ext in ['.jpg', '.bboxes.tsv', '.bboxes.labels.tsv']:
            if os.path.exists(src[:-4] + ext):
                shutil.copy(src[:-4] + ext, os.path.join(test_dir, base[:-4] + ext))
print('testImages:', len(glob.glob(os.path.join(test_dir, '*.jpg'))), 'jpg /',
      len(glob.glob(os.path.join(test_dir, '*.bboxes.tsv'))), 'annotation files')
code = '''
import sys, os
sys.path.insert(0, "/content/spike/CNTK/Examples/Image/Detection")
from utils.annotations.annotations_helper import create_class_dict, create_map_files, parse_class_map_file
data = "/content/spike/CNTK/Examples/Image/DataSets/SPIKE"
cd = create_class_dict(data)
print("class dict:", cd)
create_map_files(data, cd, True)
create_map_files(data, cd, False)
print("classes:", parse_class_map_file(os.path.join(data, "class_map.txt")))
print("test map entries:", len(open(os.path.join(data, "test_img_file.txt")).read().splitlines()))
'''
r = subprocess.run(['/content/c36/bin/python','-c',code], capture_output=True, text=True, env=ENV_LD)
print(r.stdout); print(r.stderr[-1500:]); assert r.returncode == 0

### 8. Input preview — a GSYC test image with the authors' ground-truth spike boxes

**Sample provenance:** `GSYC_test001.jpg` from the SPIKE dataset (SourceForge `SPIKE_dataset.zip`), one of the 20 held-out GSYC test images of Table 2. Green rectangles are the authors' ground-truth spike annotations (`.bboxes.tsv`, read directly — nothing is invented). The annotation rows are corner coordinates; the statistics printed below confirm the corner interpretation (as the authors' own VOC-style evaluation also assumes).／日本語：GSYC テスト画像と著者の正解スパイク箱（緑）を表示します。座標はコーナー形式であることを統計で確認します。

In [ ]:
import os, numpy as np, glob, matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
DATA = '/content/spike/CNTK/Examples/Image/DataSets/SPIKE'
img_path = os.path.join(DATA, 'testImages', 'GSYC_test001.jpg')
vals = np.loadtxt(img_path[:-4] + '.bboxes.tsv', ndmin=2)
w_corner = (vals[:,2]-vals[:,0]); h_corner = (vals[:,1+2]-vals[:,1])
print('rows:', vals.shape[0], '| corner interp: median w=%.0f h=%.0f, h/w=%.1f' % (np.median(w_corner), np.median(h_corner), np.median(h_corner/w_corner)))
img = Image.open(img_path).convert('RGB'); dr = ImageDraw.Draw(img)
for x1, y1, x2, y2 in vals:
    dr.rectangle([x1, y1, x2, y2], outline=(0, 220, 0), width=4)
plt.figure(figsize=(10, 7.5))
plt.imshow(img); plt.axis('off')
plt.title('GSYC_test001.jpg — input with authors\' ground-truth spike boxes (green)')
plt.savefig('/content/preview_gt.png', dpi=110, bbox_inches='tight'); plt.show()
print('image size:', img.size)

### 9. Load the released GSYC model through the authors' own path

`prepare()` validates the dataset layout; `train_faster_rcnn()` with the authors' default `MAKE_MODE=True` **loads the existing model** instead of training — exactly the path their `run_faster_rcnn.py` uses when a trained model exists. We point `cfg['MODEL_PATH']` at the downloaded released `GSYC_model.model` (no VGG16 base download is needed: the stored e2e model contains the full network). The ProposalLayer user function is deserialized from `utils/rpn/proposal_layer.py`.／日本語：著者のコード経由で公開済み GSYC モデルを読み込みます（学習は行いません）。

In [ ]:
import subprocess, textwrap
script = textwrap.dedent('''
    import os, sys, time, glob, json
    import numpy as np
    sys.path.insert(0, "/content/spike/CNTK/Examples/Image/Detection/FasterRCNN")
    sys.path.insert(0, "/content/spike/CNTK/Examples/Image/Detection")
    os.chdir("/content/spike/CNTK/Examples/Image/Detection/FasterRCNN")

    from FasterRCNN_config import cfg
    from utils.configs.VGG16_config import cfg as network_cfg
    from utils.configs.SPIKE_config import cfg as dataset_cfg
    from utils.config_helpers import merge_configs
    cfg = merge_configs([cfg, network_cfg, dataset_cfg])
    from FasterRCNN_train import prepare, train_faster_rcnn
    prepare(cfg, False)
    cfg['MODEL_PATH'] = '/content/GSYC_model.model'
    eval_model = train_faster_rcnn(cfg)
    print('classes:', cfg['DATA'].CLASSES, flush=True)
    print('NMS threshold:', cfg.RESULTS_NMS_THRESHOLD, '| conf threshold:', cfg.RESULTS_NMS_CONF_THRESHOLD, flush=True)

    from FasterRCNN_eval import FasterRCNN_Evaluator
    from utils.nms_wrapper import apply_nms_to_single_image_results
    from utils.proposal_helpers import compute_image_stats
    evaluator = FasterRCNN_Evaluator(eval_model, cfg)

    DATA = '/content/spike/CNTK/Examples/Image/DataSets/SPIKE'
    test_imgs = sorted(glob.glob(os.path.join(DATA, 'testImages', '*.jpg')))
    print('test images:', len(test_imgs), flush=True)

    # sequence -> filename from the map file (os.listdir order is NOT sorted)
    name2seq = {}
    for line in open(os.path.join(DATA, 'test_img_file.txt')).read().splitlines():
        parts = line.split(chr(9))
        name2seq[os.path.basename(parts[1])] = int(parts[0])
    roi_lines = open(os.path.join(DATA, 'test_roi_file.txt')).read().splitlines()
    gt_per_seq = {}
    for line in roi_lines:
        idx, rest = line.strip().split(' ', 1)
        vals = rest.replace('|roiAndLabel', '').strip().split()
        gt_per_seq[int(idx)] = np.array(vals, dtype=np.float32).reshape(-1, 5)

    from PIL import Image
    NUM_CLASSES = cfg['DATA'].NUM_CLASSES
    all_boxes = [[[] for _ in range(len(test_imgs))] for _ in range(NUM_CLASSES)]
    all_gt_infos = {c: [] for c in cfg['DATA'].CLASSES}
    per_image = []
    nms_dets_per_img = []
    for img_i, p in enumerate(test_imgs):
        t = time.time()
        regressed_rois, out_cls_pred = evaluator.process_image(p)
        labels = out_cls_pred.argmax(axis=1)
        scores = out_cls_pred.max(axis=1)
        # raw (pre-NMS) rois, exactly like the authors' compute_test_set_aps collection
        csl = np.hstack((regressed_rois, scores.reshape(-1, 1), labels.reshape(-1, 1)))
        for cls_j in range(1, NUM_CLASSES):
            sel = csl[csl[:, -1] == cls_j]
            all_boxes[cls_j][img_i] = sel[:, :-1].astype(np.float32, copy=False)
        # per-image detections after the authors' NMS (their plot path), paper's C = 0.5
        keep = apply_nms_to_single_image_results(
            regressed_rois, labels, scores,
            cfg.USE_GPU_NMS, cfg.GPU_ID,
            cfg.RESULTS_NMS_THRESHOLD, cfg.RESULTS_NMS_CONF_THRESHOLD)
        keep1 = [i for i in keep if labels[i] == 1]
        d05 = np.array([[*regressed_rois[i], scores[i]] for i in keep1 if scores[i] >= 0.5], dtype=np.float32)
        nms_dets_per_img.append(d05)
        # ground truth scaled into the padded-canvas space, mirroring od_reader (annotations *= scale_factor)
        w, h = Image.open(p).size
        img_stats = compute_image_stats(w, h, cfg.IMAGE_WIDTH, cfg.IMAGE_HEIGHT)
        scale, top, left = img_stats[-1], img_stats[4], img_stats[6]
        gt = gt_per_seq[name2seq[os.path.basename(p)]]
        fg = gt[gt[:, -1] == 1].copy()
        fg[:, :4] = np.round(fg[:, :4] * scale + (left, top, left, top))
        all_gt_infos[cfg['DATA'].CLASSES[1]].append({'bbox': fg, 'difficult': [False] * len(fg), 'det': [False] * len(fg)})
        rec = {'file': os.path.basename(p), 'gt': int(len(fg)),
               'n_conf05': int(len(d05)), 'n_conf0': len(keep1), 'time_s': round(time.time() - t, 1)}
        per_image.append(rec)
        print('%(file)s: GT=%(gt)d detected(post-NMS,conf>=0.5)=%(n_conf05)d [%(time_s).1fs]' % rec, flush=True)

    import pickle
    with open('/content/detections.pkl', 'wb') as f:
        pickle.dump({'all_boxes': all_boxes, 'all_gt_infos': all_gt_infos,
                     'classes': cfg['DATA'].CLASSES, 'per_image': per_image,
                     'nms_dets_per_img': nms_dets_per_img}, f)
    print('inference done', flush=True)
''')
open('/content/final_eval.py', 'w').write(script)
r = subprocess.run(['/content/c36/bin/python', '/content/final_eval.py'], capture_output=True, text=True, env=ENV_LD)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
assert r.returncode == 0, 'inference failed'


### 10. Detection overlays — input vs ground truth vs model output

Two representative test images: **green = authors' ground-truth spike boxes**, **red = released GSYC model detections** after the authors' NMS (threshold 0.35) and confidence ≥ 0.5 (the paper's C = 0.5). These are actual outputs of this run, rendered from the real test images.／日本語：緑＝正解ボックス、赤＝モデルの検出結果（NMS 0.35、信頼度 ≥ 0.5）。

*Source-to-notebook mapping:* detection collection mirrors the authors' `compute_test_set_aps` (`FasterRCNN_eval.py` L78–116); overlay drawing is AI-generated glue using the authors' `regressed_rois`/`cls_pred` outputs.

In [ ]:
import numpy as np, os, glob, pickle, matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
DATA = '/content/spike/CNTK/Examples/Image/DataSets/SPIKE'
with open('/content/detections.pkl', 'rb') as f:
    det = pickle.load(f)
per_image = {r['file']: r for r in det['per_image']}
fig, axes = plt.subplots(2, 1, figsize=(10, 15))
for ax, name in zip(axes, ['GSYC_test001.jpg', 'GSYC_test012.jpg']):
    p = os.path.join(DATA, 'testImages', name)
    img = Image.open(p).convert('RGB'); dr = ImageDraw.Draw(img)
    vals = np.loadtxt(p[:-4] + '.bboxes.tsv', ndmin=2)
    for x1, y1, x2, y2 in vals:
        dr.rectangle([x1, y1, x2, y2], outline=(0, 220, 0), width=4)
    img_index = [i for i, r in enumerate(det['per_image']) if r['file'] == name][0]
    rois = det['all_boxes'][1][img_index]
    scores = rois[:, 4]; keep = scores >= 0.5
    W, H = img.size
    sc = 850.0 / max(W, H)
    _top = (850 - int(round(H * sc))) // 2
    _left = (850 - int(round(W * sc))) // 2
    d_orig = (rois[keep][:, :4] - [_left, _top, _left, _top]) / sc
    for x1, y1, x2, y2 in d_orig:
        dr.rectangle([x1, y1, x2, y2], outline=(255, 40, 40), width=4)
    ax.imshow(img); ax.axis('off')
    r = per_image[name]
    ax.set_title('%s — GT %d (green) vs detected %d (red, conf>=0.5)' % (name, r['gt'], r['n_conf05']))
plt.tight_layout(); plt.savefig('/content/overlays.png', dpi=100, bbox_inches='tight'); plt.show()

### 11. Validation metrics with the authors' evaluation code

**mAP** is computed with the authors' `evaluate_detections` (VOC-style, IoU 0.5, their NMS at 0.35, `RESULTS_NMS_CONF_THRESHOLD = 0.0`) — both the continuous AP and the 11-point variant the paper text mentions. **TP/FP/FN, precision, recall, accuracy, F1** are computed with a small AI-generated matcher that mirrors the authors' `_voc_computePrecisionRecallAp` greedy matching (documented glue); the paper defines Accuracy = (TP+TN)/(TP+TN+FP+FN) with TN always 0, i.e. TP/(TP+FP+FN). Reference (Table 2): TP 1463, FP 41, FN 107 over 1570 GT, mAP 0.6653.／日本語：著者の評価コードで mAP を、そのマッチング規則に従う補助コードで TP/FP/FN を計算し、Table 2 の参照値と比較します。

In [ ]:
import subprocess, textwrap
script = textwrap.dedent('''
    import sys, os, json, pickle
    import numpy as np
    sys.path.insert(0, "/content/spike/CNTK/Examples/Image/Detection")
    os.chdir("/content/spike/CNTK/Examples/Image/Detection/FasterRCNN")
    with open('/content/detections.pkl', 'rb') as f:
        det = pickle.load(f)
    from utils.map_helpers import evaluate_detections
    aps = evaluate_detections(det['all_boxes'], det['all_gt_infos'], det['classes'],
                              use_gpu_nms=False, device_id=0,
                              nms_threshold=0.35, conf_threshold=0.0, use_07_metric=False)
    aps07 = evaluate_detections(det['all_boxes'], det['all_gt_infos'], det['classes'],
                                use_gpu_nms=False, device_id=0,
                                nms_threshold=0.35, conf_threshold=0.0, use_07_metric=True)
    print('mAP (continuous VOC AP):', aps, flush=True)
    print('mAP (11-point):', aps07, flush=True)

    # AI-glue matcher mirroring the authors' _voc_computePrecisionRecallAp (IoU 0.5, greedy by confidence)
    def match_image(dets, gt_boxes, iou_thresh=0.5):
        dets = np.asarray(dets, dtype=np.float32).reshape(-1, 5)
        gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
        tp = np.zeros(len(dets)); fp = np.zeros(len(dets))
        matched = np.zeros(len(gt_boxes), dtype=bool)
        for d in np.argsort(-dets[:, -1]):
            bb = dets[d, :4]
            if len(gt_boxes):
                ixmin = np.maximum(gt_boxes[:, 0], bb[0]); iymin = np.maximum(gt_boxes[:, 1], bb[1])
                ixmax = np.minimum(gt_boxes[:, 2], bb[2]); iymax = np.minimum(gt_boxes[:, 3], bb[3])
                iw = np.maximum(ixmax - ixmin + 1., 0.); ih = np.maximum(iymax - iymin + 1., 0.)
                inter = iw * ih
                ag = (gt_boxes[:, 2] - gt_boxes[:, 0] + 1.) * (gt_boxes[:, 3] - gt_boxes[:, 1] + 1.)
                ab = (bb[2] - bb[0] + 1.) * (bb[3] - bb[1] + 1.)
                iou = inter / (ag + ab - inter)
                j = int(np.argmax(iou))
                if iou[j] >= iou_thresh and not matched[j]:
                    tp[d] = 1; matched[j] = True
                else:
                    fp[d] = 1
            else:
                fp[d] = 1
        ntp, nfp = int(tp.sum()), int(fp.sum())
        return ntp, nfp, len(gt_boxes) - ntp

    per = []
    gt_total = tp_total = fp_total = fn_total = 0
    for i, name in enumerate([r['file'] for r in det['per_image']]):
        dets = det['nms_dets_per_img'][i]  # post-NMS detections at the paper's C = 0.5
        gt_boxes = det['all_gt_infos']['S'][i]['bbox'][:, :4]
        ntp, nfp, nfn = match_image(dets, gt_boxes)
        per.append({'file': name, 'gt': int(len(gt_boxes)), 'tp': ntp, 'fp': nfp, 'fn': nfn})
        gt_total += len(gt_boxes); tp_total += ntp; fp_total += nfp; fn_total += nfn
    prec = tp_total / (tp_total + fp_total) if tp_total + fp_total else 0.0
    rec = tp_total / (tp_total + fn_total) if tp_total + fn_total else 0.0
    acc = tp_total / (tp_total + fp_total + fn_total)
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    metrics = {'per_image': per, 'gt_total': gt_total, 'tp': tp_total, 'fp': fp_total, 'fn': fn_total,
               'precision': round(prec, 4), 'recall': round(rec, 4),
               'accuracy_TN0': round(acc, 4), 'f1': round(f1, 4),
               'mAP_continuous': aps['S'], 'mAP_11point': aps07['S']}
    json.dump(metrics, open('/content/metrics.json', 'w'), indent=1)
    print(json.dumps({k: v for k, v in metrics.items() if k != 'per_image'}, indent=1), flush=True)
''')
open('/content/final_metrics.py', 'w').write(script)
r = subprocess.run(['/content/c36/bin/python', '/content/final_metrics.py'], capture_output=True, text=True, env=ENV_LD)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
assert r.returncode == 0, 'metrics failed'


### 12. Per-image comparison with the paper's Table 2

Our measured per-image TP/FP/FN are compared with the paper's Table 2 rows (`Test_XXX` ↔ `GSYC_testXXX`, reference values extracted from the open-access article XML). Small deviations are expected from the confidence-threshold convention (the paper counts "Detected" at C = 0.5; the mAP pipeline of the released code uses confidence 0.0). A CSV with the full comparison is exported to `/content/spike_eval_results.csv`.／日本語：測定結果を論文 Table 2 の行と比較し、CSV に出力します。

In [ ]:
import json, csv, matplotlib
import matplotlib.pyplot as plt
metrics = json.load(open('/content/metrics.json'))
table2 = {"Test_001": (73, 71, 70, 1, 3), "Test_012": (75, 68, 68, 0, 7), "Test_025": (87, 85, 84, 1, 3), "Test_032": (80, 76, 76, 0, 4), "Test_118": (76, 73, 70, 3, 6), "Test_141": (66, 61, 58, 3, 8), "Test_185": (69, 68, 65, 3, 4), "Test_199": (72, 69, 68, 1, 4), "Test_220": (80, 79, 76, 3, 4), "Test_242": (70, 64, 63, 1, 7), "Test_254": (83, 77, 76, 1, 7), "Test_320": (80, 77, 74, 3, 6), "Test_383": (87, 84, 78, 6, 9), "Test_399": (80, 78, 77, 1, 3), "Test_417": (96, 93, 89, 4, 7), "Test_421": (71, 73, 70, 3, 1), "Test_422": (82, 79, 78, 1, 4), "Test_432": (85, 81, 79, 2, 6), "Test_437": (70, 64, 62, 2, 8), "Test_480": (88, 84, 82, 2, 6)}
rows = []
for r in metrics['per_image']:
    num = ''.join(ch for ch in r['file'] if ch.isdigit()).lstrip('0')
    key = 'Test_' + num.zfill(3)
    ref = table2.get(key, (None,)*5)
    rows.append({'file': r['file'], 'GT': r['gt'], 'TP_measured': r['tp'], 'FP_measured': r['fp'], 'FN_measured': r['fn'],
                 'GT_paper': ref[0], 'TP_paper': ref[2], 'FP_paper': ref[3], 'FN_paper': ref[4]})
with open('/content/spike_eval_results.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print('totals: GT %d (paper 1570), TP %d (paper 1463), FP %d (paper 41), FN %d (paper 107)' %
      (metrics['gt_total'], metrics['tp'], metrics['fp'], metrics['fn']))
print('mAP continuous %.4f (paper 0.6653) | mAP 11-point %.4f | accuracy(TN=0) %.4f | precision %.4f | F1 %.4f' %
      (metrics['mAP_continuous'], metrics['mAP_11point'], metrics['accuracy_TN0'], metrics['precision'], metrics['f1']))
fig, ax = plt.subplots(figsize=(11, 4.5))
idx = range(len(rows))
ax.bar([i - 0.2 for i in idx], [r['GT'] for r in rows], width=0.4, label='GT (authors\' annotations)')
ax.bar([i + 0.2 for i in idx], [r['TP_measured'] for r in rows], width=0.4, label='TP measured (IoU 0.5)')
ax.set_xticks(list(idx)); ax.set_xticklabels([r['file'].replace('GSYC_test', '#') for r in rows], rotation=45, fontsize=8)
ax.set_xlabel('GSYC test image'); ax.set_ylabel('spike count'); ax.legend()
ax.set_title('Per-image ground truth vs true positives (this run; additional visualization, not an author figure)')
plt.tight_layout(); plt.savefig('/content/counts_graph.png', dpi=110); plt.show()

## Interpretation, differences, validation record

**What was executed:** the released GSYC Faster R-CNN checkpoint was loaded with the authors' own CNTK 2.7 evaluation code and run over the paper's complete 20-image GSYC test set on a Colab **CPU** runtime; all downloads and computation happened inside this notebook.

**What to compare:** total TP/FP/FN against Table 2 (paper: 1463/41/107 over 1570 ground-truth spikes), mAP against 0.6653 (the released code computes the continuous VOC AP; the paper text additionally mentions the 11-point variant), and per-image detected counts. The paper's per-image "Detected" applies its confidence threshold C = 0.5 — our `n_conf05` column follows that convention, while the mAP path follows the released code's `RESULTS_NMS_CONF_THRESHOLD = 0.0`.

**Known differences / limitations:** no GPU path exists for this checkpoint on modern Colab hardware (CNTK 2.7 = CUDA 9 era); inference is CPU-bound (~12 s/image). The SourceForge project states no license for the dataset/weights; the paper is CC BY 4.0 and CNTK example code is MIT. This single-model demonstration does not verify the paper's other models (+GSGC, +YSYC, GSYC++), the spike-density analysis, or dataset-level claims beyond Table 2.

**References:** David et al., Plant Methods 2018 (PMC6236889); erratum 2019 (PMC6425585); SourceForge `spike-dataset` project (pbrc); Microsoft CNTK v2.7, commit `ae9c9c7c5f9e6072cc9c94c254f816dbdc1c5be6` (MIT). PhenoPaper investigation used only as prior research guidance.／日本語：公開済み GSYC モデルを著者の評価コードで 20 枚のテスト画像に再評価しました。GPU 非対応（CNTK 2.7 は CUDA 9 時代）のため CPU で実行しています。

**Data discrepancy noted during validation:** the released test annotations contain fewer ground-truth boxes than the paper's Table 2 GT-count column (printed `gt_total` here vs the reference 1570); per-image GT counts also differ irregularly. All metrics in this notebook are computed against the released annotations, so deviations from Table 2 are expected in addition to model/threshold variation. The mAP is the released code's continuous VOC AP; the paper text additionally mentions an 11-point variant, which the released helper reported as 0.0 on this data (see printed output).／日本語：公開アノテーションは論文 Table 2 の GT 数より少ないため、数値のずれは予想されます。